<a href="https://colab.research.google.com/github/OthmaneL4/IA-y-Big-Data-/blob/main/Copia_de_sesion11_motores_reglas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S11 - Motores de reglas (Pagarium)

**Objetivo (RA5-b):** pasar del micro-motor a los motores de producción: **DMN**, **GoRules ZEN** y `rule-engine`, y medir cuándo compensa.

> Una **tabla de decisión** es una lista de filas "si se cumplen estas condiciones → esta decisión".

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre |
| `{ "clave": "valor" }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: valores ordenados |
| `def nombre(...):` | Crea una **función** |
| `"texto"` / `'texto'` | Un **texto** (string), entre comillas |
| `==` / `<` / `>=` | Comparaciones: igual / menor / mayor o igual |
| `and` / `or` | Y lógico / O lógico |
| `if ...:` / `for ... in ...:` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

Ejecuta las celdas de arriba a abajo; los comentarios `#` explican cada línea.

## 1. Política de Pagarium como tabla DMN

Una **tabla de decisión** cruza las condiciones con una acción; el guion `—` significa «no importa».

**Completa la columna Decisión** (la primera fila ya está resuelta):

| Importe | Antigüedad (meses) | Intentos | Decisión |
|---|---|---|---|
| `< 500` | `>= 6` | `< 4` | **Aprobar** |
| `< 500` | `>= 6` | `>= 4` | … |
| `>= 500` | — | — | … |
| — | `< 6` | — | … |

*Pista:* la fila 3 (`importe >= 500`) y la fila 4 (`antigüedad < 6`) se solapan cuando se cumplen las dos a la vez.

## 2. `rule-engine`: la política de Pagarium paso a paso

Vamos a **traducir la tabla DMN a código**, pieza a pieza. Completa cada `# TODO` y ejecuta.

### Paso 1 · El contexto (los hechos)

In [1]:
%pip install rule-engine
import rule_engine

# El contexto: un diccionario con los hechos del pago (parejas nombre -> valor).
ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}

### Paso 2 · La regla de aprobar

`Rule(...)` guarda una condición en texto; `.matches(ctx)` devuelve `True` si se cumple con esos hechos.

In [5]:
aprueba = rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos < 4")
print("aprueba:", aprueba.matches(ctx))   # False: n_intentos=5 no es < 4

aprueba: False


### Paso 3 · Las otras tres reglas (TODO)

Escribe las tres reglas que faltan de la tabla DMN (fila 2: revisar por intentos; fila 3: revisar por importe; fila 4: rechazar).

In [17]:
#  escribe las tres reglas que faltan (usa and / or / < / >=)
revisar_intentos = rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos >= 4")   # fila 2: importe < 500 y antiguedad >= 6 y n_intentos >= 4
revisar_importe  = rule_engine.Rule("importe >= 500")   # fila 3: importe >= 500
rechazar         = rule_engine.Rule("antiguedad_meses < 6")   # fila 4: antiguedad < 6

### Paso 4 · Probar los casos y ver el solape (TODO)

In [18]:
reglas = {"aprobar": aprueba, "revisar_intentos": revisar_intentos,
          "revisar_importe": revisar_importe, "rechazar": rechazar}

def decidir(c):
    # TODO: devuelve los nombres de las reglas que encajan en c
    return [nombre for nombre, r in reglas.items() if r.matches(c)]

casos = [{"importe": 600, "antiguedad_meses": 3, "n_intentos": 2},
         {"importe": 100, "antiguedad_meses": 12, "n_intentos": 1},
         {"importe": 1000, "antiguedad_meses": 5, "n_intentos": 5}]
for c in casos:
    print(c, "->", decidir(c))

solape = {"importe": 600, "antiguedad_meses" : 3, "n_intentos": 2}
print(solape, "->", decidir(solape))
# TODO: prueba los 4 casos (uno por fila) y este caso de SOLAPE:
#   {"importe": 600, "antiguedad_meses": 3, "n_intentos": 2}

{'importe': 600, 'antiguedad_meses': 3, 'n_intentos': 2} -> ['revisar_importe', 'rechazar']
{'importe': 100, 'antiguedad_meses': 12, 'n_intentos': 1} -> ['aprobar']
{'importe': 1000, 'antiguedad_meses': 5, 'n_intentos': 5} -> ['revisar_importe', 'rechazar']
{'importe': 600, 'antiguedad_meses': 3, 'n_intentos': 2} -> ['revisar_importe', 'rechazar']


### Otro ejemplo: descuentos por categoría

In [ ]:
reglas = {
    "descuento_20": rule_engine.Rule("categoria == 'vip' and importe >= 100"),
    "descuento_10": rule_engine.Rule("categoria == 'vip' and importe < 100"),
    "descuento_5":  rule_engine.Rule("categoria == 'normal' and importe >= 100"),
    "sin_descuento": rule_engine.Rule("categoria == 'normal' and importe < 100"),
}
def decidir2(c):
    return [n for n, r in reglas.items() if r.matches(c)]
for c in [{"categoria": "vip", "importe": 200}, {"categoria": "normal", "importe": 50}]:
    print(c, "->", decidir2(c))

## 4. GoRules ZEN (fintech)

ZEN ejecuta un grafo de decisión (formato **JDM**, JSON). Lo montamos paso a paso.

### Paso 1 · El motor

In [ ]:
%pip install zen-engine
import json
from zen import ZenEngine

engine = ZenEngine()   # el motor que evalúa grafos JDM

### Paso 2 · Los nodos del grafo (con las reglas por rellenar)

Un grafo tiene **nodos** (entrada → tabla → salida). La tabla lleva `hitPolicy`, `inputs`, `outputs` y `rules`.

In [ ]:
graph = {
    "contentType": "application/vnd.gorules.decision",
    "nodes": [
        {"id": "input", "type": "inputNode", "name": "Request", "position": {"x": 0, "y": 0}},
        {"id": "table", "type": "decisionTableNode", "name": "Politica",
         "content": {
             "hitPolicy": "first",   # gana la primera fila que encaja
             "inputs":  [{"id": "i1", "field": "importe", "name": "Importe", "ref": ""}],
             "outputs": [{"id": "o1", "field": "decision", "name": "Decision", "ref": ""}],
             "rules": [
                 # TODO: escribe las reglas (condición -> decisión, como texto JSON)
             ],
         },
         "position": {"x": 200, "y": 0}},
        {"id": "out", "type": "outputNode", "name": "Response",
         "content": {"fields": [{"id": "decision", "field": "decision"}]},
         "position": {"x": 400, "y": 0}},
    ],
    "edges": [
        # TODO: conecta input -> table -> out
    ],
}

### Paso 3 · Rellenar las reglas y las aristas (TODO)

Las **reglas** son dicts con la condición (`i1`) y la decisión (`o1`, como texto JSON entre comillas); las **aristas** unen los nodos por su `id`.

In [ ]:
# rules:
#   {"i1": "< 500",  "_id": "r1", "o1": "\"aprobar\""}
#   {"i1": ">= 500", "_id": "r2", "o1": "\"revisar\""}
# edges:
#   {"id": "e1", "sourceId": "input", "targetId": "table"}
#   {"id": "e2", "sourceId": "table", "targetId": "out"}

### Paso 4 · Ejecutar

In [ ]:
decision = engine.create_decision(json.dumps(graph))   # json.dumps pasa el dict a texto JSON
print(decision.evaluate(json.dumps({"importe": 200})))  # evalúa con importe 200

### Grafo ZEN con dos entradas

La tabla ahora decide por importe **y** número de intentos.

In [ ]:
graph2 = {
  "contentType": "application/vnd.gorules.decision",
  "nodes": [
    {"id": "input", "type": "inputNode", "name": "Request", "position": {"x": 0, "y": 0}},
    {"id": "table", "type": "decisionTableNode", "name": "Politica",
     "content": {
       "hitPolicy": "first",
       "inputs": [
         {"id": "i1", "field": "importe", "name": "Importe", "ref": ""},
         {"id": "i2", "field": "n_intentos", "name": "Intentos", "ref": ""},
       ],
       "outputs": [{"id": "o1", "field": "decision", "name": "Decision", "ref": ""}],
       "rules": [
         {"_id": "r1", "i1": "< 500", "i2": "< 4", "o1": "\"aprobar\""},
         {"_id": "r2", "i1": "< 500", "i2": ">= 4", "o1": "\"revisar\""},
         {"_id": "r3", "i1": ">= 500", "i2": "-", "o1": "\"revisar\""},   # "-" = no importa
       ],
     },
     "position": {"x": 200, "y": 0}},
    {"id": "out", "type": "outputNode", "name": "Response",
     "content": {"fields": [{"id": "decision", "field": "decision"}]},
     "position": {"x": 400, "y": 0}},
  ],
  "edges": [
    {"id": "e1", "sourceId": "input", "targetId": "table"},
    {"id": "e2", "sourceId": "table", "targetId": "out"},
  ],
}
decision2 = engine.create_decision(json.dumps(graph2))
print(decision2.evaluate(json.dumps({"importe": 200, "n_intentos": 2})))   # -> aprobar
print(decision2.evaluate(json.dumps({"importe": 200, "n_intentos": 5})))   # -> revisar

## 5. Benchmark: ¿compensa un motor?

Un motor de reglas es más cómodo pero **mucho más lento** que un `if/else`. Medimos cuánto.

In [ ]:
import timeit   # timeit mide el tiempo de ejecutar algo muchas veces

ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}

def ifelse(c):                        # la misma decisión, escrita a mano con if/else
    if c["importe"] < 500 and c["antiguedad_meses"] >= 6 and c["n_intentos"] < 4:
        return "aprobar"
    if c["n_intentos"] >= 4 or c["antiguedad_meses"] < 6:
        return "rechazar"
    return "revisar"

# timeit.timeit(ejecuta la función 100000 veces) / 100000 = tiempo medio; * 1e6 = microsegundos
print("if/else:", round(timeit.timeit(lambda: ifelse(ctx), number=100000) / 100000 * 1e6, 3), "us")

**Actividad:** mide con `timeit` tu `if/else` frente a `rule-engine` para el mismo caso y calcula el factor (cuántas veces es más lento el motor).